# Spike — joignabilité des sources sur le département 33 (Gironde)

Objectif (cf. [ADR 0001](../docs/adr/0001-valider-joignabilite-avant-de-figer-le-socle.md)) :
**mesurer les taux de match réels entre DVF, RNB, DPE et BAN avant de figer le pivot et le périmètre.**

Modèle de jointure validé via le catalogue data.gouv (cf. [SOURCES_DONNEES.md](../docs/SOURCES_DONNEES.md)) :

- `DVF.id_parcelle` == `RNB.plots[].id` == `BAN.cad_parcelles`  (parcelle)
- `DPE.identifiant_ban` == `RNB.addresses[].cle_interop_ban` == `BAN.id`  (adresse, même namespace BAN cle_interop)
- Le DPE open data ne porte **ni rnb_id ni parcelle** → le lien bâtiment passe par la clé BAN (ou le spatial).

Règles de nettoyage : **tout en String** (préserve les zéros de `code_postal`/`code_commune`/`id_parcelle`) ; dates/prix castés plus tard.
Traitement & écriture en **Polars** ; requêtes de mesure en **DuckDB**.


In [1]:
import polars as pl
import duckdb
import json, zipfile, urllib.request, urllib.parse
from pathlib import Path

DEPT = "33"
YEARS = range(2021, 2026)  # geo-dvf latest: 2021-2025
ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "pyproject.toml").exists())
RAW = ROOT / "data/raw"; RAW.mkdir(parents=True, exist_ok=True)
INTERIM = ROOT / "data/interim"; INTERIM.mkdir(parents=True, exist_ok=True)
pl.Config.set_tbl_rows(15); pl.Config.set_fmt_str_lengths(60)

def download(url, dest):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"✓ déjà là: {dest.name} ({dest.stat().st_size/1e6:.1f} Mo)"); return dest
    print(f"⤓ {url}")
    urllib.request.urlretrieve(url, dest)
    print(f"  → {dest.name} ({dest.stat().st_size/1e6:.1f} Mo)"); return dest


## 1. DVF géolocalisé (dept 33, 2020-2025)

In [2]:
dvf_cache = INTERIM / f"dvf_{DEPT}.parquet"
if dvf_cache.exists():
    dvf = pl.read_parquet(dvf_cache)
else:
    frames = []
    for y in YEARS:
        url = f"https://files.data.gouv.fr/geo-dvf/latest/csv/{y}/departements/{DEPT}.csv.gz"
        f = download(url, RAW / f"dvf_{DEPT}_{y}.csv.gz")
        frames.append(pl.read_csv(f, separator=",", infer_schema=False))  # tout en String
    dvf = pl.concat(frames, how="diagonal_relaxed")
    dvf.write_parquet(dvf_cache)
print("DVF:", dvf.shape)
dvf.select("id_mutation","date_mutation","type_local","surface_reelle_bati",
           "nombre_pieces_principales","id_parcelle","code_postal","code_commune",
           "valeur_fonciere","longitude","latitude").head()


DVF: (525352, 40)


id_mutation,date_mutation,type_local,surface_reelle_bati,nombre_pieces_principales,id_parcelle,code_postal,code_commune,valeur_fonciere,longitude,latitude
str,str,str,str,str,str,str,str,str,str,str
"""2021-502604""","""2021-01-04""",null,null,null,"""33192000AE0895""","""33170""","""33192""","""235000""","""-0.594056""","""44.777138"""
"""2021-502605""","""2021-01-06""","""Appartement""","""19""","""1""","""33318000DI0167""","""33600""","""33318""","""103950""","""-0.606873""","""44.794243"""
"""2021-502606""","""2021-01-05""","""Dépendance""",null,"""0""","""33063000DR0384""","""33000""","""33063""","""214380""","""-0.570149""","""44.832857"""
"""2021-502606""","""2021-01-05""","""Appartement""","""38""","""2""","""33063000DR0384""","""33000""","""33063""","""214380""","""-0.570149""","""44.832857"""
"""2021-502607""","""2021-01-08""","""Appartement""","""63""","""3""","""33162000AC0426""","""33320""","""33162""","""177000""","""-0.64531""","""44.889137"""


## 2. Nettoyage adresses — règle commune

Tout reste String (codes préservés). On ajoute une normalisation **texte** (minuscule, espaces) pour le matching de voie ;
le repli accents sera ajouté plus tard.

In [3]:
def norm_addr(col: pl.Expr) -> pl.Expr:
    return (col.cast(pl.String).str.strip_chars()
               .str.replace_all(r"\s+", " ").str.to_lowercase())

dvf = dvf.with_columns(voie_norm=norm_addr(pl.col("adresse_nom_voie")))
dvf.select("adresse_nom_voie","voie_norm","code_postal","code_commune","id_parcelle").head()


adresse_nom_voie,voie_norm,code_postal,code_commune,id_parcelle
str,str,str,str,str
"""RUE SAINT FRANCOIS XAVIER""","""rue saint francois xavier""","""33170""","""33192""","""33192000AE0895"""
"""RUE DU RELAIS""","""rue du relais""","""33600""","""33318""","""33318000DI0167"""
"""RUE DE CAUSSEROUGE""","""rue de causserouge""","""33000""","""33063""","""33063000DR0384"""
"""RUE BRAGARD""","""rue bragard""","""33000""","""33063""","""33063000DR0384"""
"""AV DU MEDOC""","""av du medoc""","""33320""","""33162""","""33162000AC0426"""


## 3. RNB (dept 33) — pivot bâtiment
Colonnes JSON `addresses` (→ cle_interop_ban) et `plots` (→ parcelle + ratio de recouvrement).

In [4]:
rnb_zip = download(
    f"https://rnb-opendata.s3.fr-par.scw.cloud/files/RNB_{DEPT}.csv.zip",
    RAW / f"RNB_{DEPT}.csv.zip")
plots_cache = INTERIM / f"rnb_plots_{DEPT}.parquet"
adr_cache   = INTERIM / f"rnb_adr_{DEPT}.parquet"
if plots_cache.exists() and adr_cache.exists():
    rnb = None
    print("RNB: caches plots/adr présents -> parsing du CSV sauté")
else:
    with zipfile.ZipFile(rnb_zip) as z:
        name = next(n for n in z.namelist() if n.endswith(".csv"))
        rnb = pl.read_csv(z.read(name), separator=";", infer_schema=False)
    print("RNB:", rnb.shape, "| colonnes:", rnb.columns)


✓ déjà là: RNB_33.csv.zip (290.8 Mo)


RNB: (1134596, 7) | colonnes: ['rnb_id', 'point', 'shape', 'status', 'ext_ids', 'addresses', 'plots']


In [5]:
# Décodage JSON — schéma minimal (json_decode exige un dtype) + cache parquet
def jcol(name):
    c = pl.col(name)
    return pl.when(c.is_null() | (c.str.strip_chars() == "")).then(pl.lit("[]")).otherwise(c)

if plots_cache.exists() and adr_cache.exists():
    rnb_plots = pl.read_parquet(plots_cache)
    rnb_adr   = pl.read_parquet(adr_cache)
else:
    PLOTS_T = pl.List(pl.Struct({"id": pl.String, "bdg_cover_ratio": pl.Float64}))
    ADDR_T  = pl.List(pl.Struct({"cle_interop_ban": pl.String}))
    rnb_plots = (rnb.select("rnb_id", jcol("plots").str.json_decode(PLOTS_T).alias("p"))
                   .explode("p").drop_nulls("p").unnest("p")
                   .rename({"id": "id_parcelle"})
                   .select("rnb_id", "id_parcelle", "bdg_cover_ratio"))
    rnb_adr = (rnb.select("rnb_id", jcol("addresses").str.json_decode(ADDR_T).alias("a"))
                  .explode("a").drop_nulls("a").unnest("a")
                  .select("rnb_id", "cle_interop_ban"))
    rnb_plots.write_parquet(plots_cache); rnb_adr.write_parquet(adr_cache)

print("rnb_plots:", rnb_plots.shape, "| rnb_adr:", rnb_adr.shape)
display(rnb_plots.head(3)); display(rnb_adr.head(3))


rnb_plots: (1883069, 3) | rnb_adr: (1168249, 2)


rnb_id,id_parcelle,bdg_cover_ratio
str,str,f64
"""11115G39BK9D""","""33380267ZD0233""",0.987663
"""1111FVEVGNYX""","""33424000WW0037""",0.999957
"""1111FVEVGNYX""","""33424000WW0038""",0.000043


rnb_id,cle_interop_ban
str,str
"""11115G39BK9D""","""33380_0078_00790"""
"""1111FVEVGNYX""","""33424_8umc5d_00008"""
"""1111FVEVGNYX""","""33424_8umc5d_00797"""


## 4. DPE logements existants (dept 33) — API ADEME
Filtrage `qs=code_departement_ban:33`, champs sélectionnés, pagination par curseur `next`, cache parquet.

In [6]:
DPE_FIELDS = ["numero_dpe","identifiant_ban","code_departement_ban","code_insee_ban",
              "code_postal_ban","nom_commune_ban","nom_rue_ban","numero_voie_ban",
              "type_batiment","etiquette_dpe","etiquette_ges","score_ban","_geopoint"]

def fetch_dpe(dept):
    base = "https://data.ademe.fr/data-fair/api/v1/datasets/dpe03existant/lines"
    q = {"size": 10000, "select": ",".join(DPE_FIELDS), "qs": f"code_departement_ban:{dept}"}
    url = base + "?" + urllib.parse.urlencode(q)
    rows = []
    while url:
        with urllib.request.urlopen(url) as r:
            d = json.load(r)
        rows.extend(d.get("results", []))
        print(f"  +{len(d.get('results', []))} (total {len(rows)}/{d.get('total')})")
        url = d.get("next")
    return pl.DataFrame(rows, infer_schema_length=None)

dpe_cache = INTERIM / f"dpe_{DEPT}.parquet"
if dpe_cache.exists():
    dpe = pl.read_parquet(dpe_cache)
else:
    dpe = fetch_dpe(DEPT); dpe.write_parquet(dpe_cache)
print("DPE:", dpe.shape)
dpe.head(3)


DPE: (376409, 14)


nom_commune_ban,etiquette_ges,identifiant_ban,code_departement_ban,code_insee_ban,_geopoint,nom_rue_ban,score_ban,type_batiment,numero_dpe,code_postal_ban,etiquette_dpe,_score,numero_voie_ban
str,str,str,str,str,str,str,f64,str,str,str,str,null,str
"""Saint-Maixant""","""A""","""33438_b1r8ky""","""33""","""33438""","""44.570591997699445,-0.25052195755074064""","""Fleur""",0.42,"""maison""","""2633E0007887V""","""33490""","""C""",null,null
"""Bordeaux""","""B""","""33063_9315_00051""","""33""","""33063""","""44.835654962321286,-0.5686819414570379""","""Cours Victor Hugo""",0.82,"""appartement""","""2633E0027829V""","""33000""","""C""",null,"""51"""
"""Lormont""","""B""","""33249_0271_00001""","""33""","""33249""","""44.87169000094878,-0.5130829687651258""","""Rue Elisee Reclus""",0.96,"""appartement""","""2633E0030346Q""","""33310""","""D""",null,"""1"""


## 5. Mesures de joignabilité (DuckDB)

In [7]:
con = duckdb.connect()
for nm, df in {"dvf": dvf, "rnb_adr": rnb_adr, "rnb_plots": rnb_plots, "dpe": dpe}.items():
    con.register(nm, df)


### M1 — Qualité de la clé adresse du DPE

In [8]:
con.sql('''
SELECT count(*) AS dpe_total,
       round(100.0*count(identifiant_ban)/count(*),1) AS pct_ban_rempli,
       round(100.0*sum(CASE WHEN regexp_matches(identifiant_ban,'_[0-9]+$') THEN 1 ELSE 0 END)
             /count(*),1) AS pct_cle_avec_numero,
       round(avg(TRY_CAST(score_ban AS DOUBLE)),3) AS score_ban_moyen
FROM dpe
''').show()


┌───────────┬────────────────┬─────────────────────┬─────────────────┐
│ dpe_total │ pct_ban_rempli │ pct_cle_avec_numero │ score_ban_moyen │
│   int64   │     double     │       double        │     double      │
├───────────┼────────────────┼─────────────────────┼─────────────────┤
│    376409 │          100.0 │                93.1 │           0.664 │
└───────────┴────────────────┴─────────────────────┴─────────────────┘



### M2 — DPE.identifiant_ban ↔ RNB.cle_interop_ban (jointure directe par clé)

In [9]:
con.sql('''
WITH d AS (SELECT DISTINCT identifiant_ban FROM dpe WHERE identifiant_ban IS NOT NULL),
     r AS (SELECT DISTINCT cle_interop_ban FROM rnb_adr)
SELECT count(*) AS dpe_cles_distinctes,
       round(100.0*sum(CASE WHEN r.cle_interop_ban IS NOT NULL THEN 1 ELSE 0 END)
             /count(*),1) AS pct_match_rnb
FROM d LEFT JOIN r ON d.identifiant_ban = r.cle_interop_ban
''').show()


┌─────────────────────┬───────────────┐
│ dpe_cles_distinctes │ pct_match_rnb │
│        int64        │    double     │
├─────────────────────┼───────────────┤
│              138397 │          87.0 │
└─────────────────────┴───────────────┘



### M3 — DVF.id_parcelle ↔ RNB.plots (couverture parcellaire)

In [10]:
con.sql('''
WITH p AS (SELECT DISTINCT id_parcelle FROM dvf WHERE id_parcelle IS NOT NULL),
     r AS (SELECT DISTINCT id_parcelle FROM rnb_plots)
SELECT count(*) AS dvf_parcelles,
       round(100.0*sum(CASE WHEN r.id_parcelle IS NOT NULL THEN 1 ELSE 0 END)
             /count(*),1) AS pct_couvertes_rnb
FROM p LEFT JOIN r ON p.id_parcelle = r.id_parcelle
''').show()


┌───────────────┬───────────────────┐
│ dvf_parcelles │ pct_couvertes_rnb │
│     int64     │      double       │
├───────────────┼───────────────────┤
│        244807 │              52.2 │
└───────────────┴───────────────────┘



### M4 — Cardinalité parcelle ↔ bâtiment (ambiguïté DVF→bâtiment)

In [11]:
con.sql('''
SELECT round(avg(n),2) AS bati_moyen_par_parcelle, max(n) AS max_bati,
       round(100.0*sum(CASE WHEN n=1 THEN 1 ELSE 0 END)/count(*),1) AS pct_parcelle_mono_bati
FROM (SELECT id_parcelle, count(DISTINCT rnb_id) AS n FROM rnb_plots GROUP BY id_parcelle)
''').show()


┌─────────────────────────┬──────────┬────────────────────────┐
│ bati_moyen_par_parcelle │ max_bati │ pct_parcelle_mono_bati │
│         double          │  int64   │         double         │
├─────────────────────────┼──────────┼────────────────────────┤
│                    2.26 │      769 │                   38.1 │
└─────────────────────────┴──────────┴────────────────────────┘



### M5 — Chaîne complète : vente DVF logement → bâtiment RNB → DPE

In [12]:
con.sql('''
WITH dvf_log AS (
        SELECT DISTINCT id_parcelle FROM dvf
        WHERE type_local IN ('Maison','Appartement') AND id_parcelle IS NOT NULL),
     parc_dpe AS (
        SELECT DISTINCT p.id_parcelle
        FROM rnb_plots p
        JOIN rnb_adr a USING(rnb_id)
        JOIN (SELECT DISTINCT identifiant_ban FROM dpe) d
          ON a.cle_interop_ban = d.identifiant_ban)
SELECT count(*) AS dvf_parcelles_log,
       round(100.0*sum(CASE WHEN pd.id_parcelle IS NOT NULL THEN 1 ELSE 0 END)
             /count(*),1) AS pct_avec_dpe_via_rnb
FROM dvf_log l LEFT JOIN parc_dpe pd ON l.id_parcelle = pd.id_parcelle
''').show()


┌───────────────────┬──────────────────────┐
│ dvf_parcelles_log │ pct_avec_dpe_via_rnb │
│       int64       │        double        │
├───────────────────┼──────────────────────┤
│             83746 │                 62.3 │
└───────────────────┴──────────────────────┘



## 6. (optionnel) Apport de la BAN comme crosswalk de secours
La BAN porte `id` (== cle_interop) **et** `cad_parcelles`. On mesure si elle couvre des parcelles DVF
au-delà de ce que RNB apporte déjà.

In [13]:
ban_gz = download(
    f"https://adresse.data.gouv.fr/data/ban/adresses/latest/csv/adresses-{DEPT}.csv.gz",
    RAW / f"ban_{DEPT}.csv.gz")
ban = pl.read_csv(ban_gz, separator=";", infer_schema=False)
print("BAN:", ban.shape, "| colonnes:", ban.columns)

ban_parc = (ban.select("id", pl.col("cad_parcelles").str.split("|").alias("pc"))
               .explode("pc").drop_nulls("pc").filter(pl.col("pc") != "")
               .rename({"pc": "id_parcelle"}))
con.register("ban_parc", ban_parc)
con.sql('''
WITH p AS (SELECT DISTINCT id_parcelle FROM dvf WHERE id_parcelle IS NOT NULL),
     b AS (SELECT DISTINCT id_parcelle FROM ban_parc),
     r AS (SELECT DISTINCT id_parcelle FROM rnb_plots)
SELECT round(100.0*sum(CASE WHEN b.id_parcelle IS NOT NULL THEN 1 ELSE 0 END)/count(*),1) AS pct_dvf_dans_ban,
       round(100.0*sum(CASE WHEN r.id_parcelle IS NOT NULL THEN 1 ELSE 0 END)/count(*),1) AS pct_dvf_dans_rnb,
       round(100.0*sum(CASE WHEN b.id_parcelle IS NOT NULL OR r.id_parcelle IS NOT NULL THEN 1 ELSE 0 END)/count(*),1) AS pct_dvf_dans_ban_ou_rnb
FROM p LEFT JOIN b ON p.id_parcelle=b.id_parcelle LEFT JOIN r ON p.id_parcelle=r.id_parcelle
''').show()


⤓ https://adresse.data.gouv.fr/data/ban/adresses/latest/csv/adresses-33.csv.gz


  → ban_33.csv.gz (27.6 Mo)


BAN: (734551, 23) | colonnes: ['id', 'id_fantoir', 'numero', 'rep', 'nom_voie', 'code_postal', 'code_insee', 'nom_commune', 'code_insee_ancienne_commune', 'nom_ancienne_commune', 'x', 'y', 'lon', 'lat', 'type_position', 'alias', 'nom_ld', 'libelle_acheminement', 'nom_afnor', 'source_position', 'source_nom_voie', 'certification_commune', 'cad_parcelles']
┌──────────────────┬──────────────────┬─────────────────────────┐
│ pct_dvf_dans_ban │ pct_dvf_dans_rnb │ pct_dvf_dans_ban_ou_rnb │
│      double      │      double      │         double          │
├──────────────────┼──────────────────┼─────────────────────────┤
│             16.2 │             52.2 │                    54.5 │
└──────────────────┴──────────────────┴─────────────────────────┘



## 7. Synthèse — résultats mesurés (dept 33, exécuté 2026-06-09)

| Mesure | Résultat | Implication |
| --- | --- | --- |
| **M1** qualité clé DPE | 100% `identifiant_ban` rempli · **93,1%** avec numéro · score_ban moyen 0,66 | clé adresse DPE de bonne qualité (≈7% au niveau voie seule) |
| **M2** DPE ↔ RNB (clé adresse) | **87,0%** des clés DPE matchent `cle_interop_ban` | jointure DPE→bâtiment par clé **viable** (la crainte FANTOIR était infondée) |
| **M3** parcelles DVF ↔ RNB `plots` | 52% global, mais **Maison 96,8% / Appart 96,3%** ; **95,0% des ventes logement** | couverture parcelle **excellente sur le bâti** (52% plombé par terrains/`(vide)` à 27%) |
| **M4** cardinalité parcelle↔bâtiment | 2,26 bâti/parcelle · max 769 · **38,1% mono-bâti** | la parcelle est un lien fiable mais **ambigu pour cibler UN bâtiment** |
| **M5** ventes logement → DPE via RNB | **62,3%** | enrichissement DPE atteignable pour ~2/3 des ventes |
| **M6** apport BAN vs RNB | BAN 16,2% vs RNB 52,2% · union 54,5% (**+2,3 pts**) | BAN quasi inutile comme crosswalk parcelle → **API-only confirmé** |

### Lecture pour les décisions reportées (ADR 0001)

- **Jointure adresse fiable** : DPE↔RNB par `cle_interop_ban` à 87%, sans re-géocodage. L'enrichissement énergétique par clé est praticable.
- **Pivot** : RNB (`rnb_id`) est joignable depuis DVF (**95% des ventes logement** via parcelle) ET depuis DPE (87% via adresse) → bon **pivot bâtiment**. La **parcelle** reste un bon lien mais **ambigu à l'unité** (38% mono-bâti seulement) : insuffisante seule pour désigner le logement vendu.
- **BAN** : aucun intérêt à l'ingérer (apport +2,3 pts) → décision API-only validée par les chiffres.
- **À approfondir** : départager bâtiment/logement quand une parcelle porte N bâtiments (surface, adresse, `bdg_cover_ratio`) ; et comprendre les 13% de DPE non matchés (clés voie-seule, millésime BAN).
